# Slider 강화학습: 합성곱 신경망 버전 (Colab GPU)

MLP 버전은 탐색 횟수를 32 -> 128 로 늘려도 Rust 엔진 상대 점수가 0.19 -> 0.22 로 거의 그대로였습니다 (이긴 판 0).
병목이 탐색이 아니라 **신경망 실력**이라서, 7x7 판을 그대로 보는 **합성곱 + 잔차 신경망**(알파제로식, 64채널 x 6블록, 약 80만 개)으로 바꿉니다.

1. **엔진의 판으로 먼저 배우기**: 엔진 판을 2000판(기존 600 + 새 1400)으로 늘리고 더 오래 배웁니다. "엔진 수 맞히기" 비율을 MLP 와 비교하세요.
2. **강화학습**: 그대로 (탐색 32회, 엔진 판 25% 섞기).
3. **실력 측정**: 10번마다 Rust 엔진과 직접 대국. **이기는 판이 나오는지**가 이 실험의 기준입니다.

- 런타임: **GPU (T4 이상)**. 엔진 판 만들기(처음 한 번, 약 45분)와 엔진 대국은 CPU 를 씁니다.
- 저장: Google Drive `slider_results/alphazero/` (`ckpt_conv_*.pkl`). 끊기면 처음부터 다시 실행하면 이어서 합니다. MLP 저장 파일과 섞이지 않습니다.


In [ ]:
# ===== 설정 =====
BRANCH = "rust-port"
ARCH = "conv"
CHANNELS = 64
BLOCKS = 6
# 1. 엔진의 판 (사전학습 데이터)
ENGINE_SETS = [(600, 1), (1400, 2)]   # (판 수, seed). 첫 묶음은 MLP 노트북이 만든 파일을 그대로 쓴다. 없는 것만 만들어 Drive 에 저장
ENGINE_TIME_MS = 30
PRETRAIN_STEPS = 12000
# 2. 강화학습
GAMES = 512
SIMULATIONS = 32
MAX_LEN = 200
DRAW_THRESHOLD = 0.2       # 형세에 따른 무승부 판단 (None 이면 끔)
LR = 1e-3
TRAIN_STEPS = 200
MINIBATCH = 1024
ENGINE_MIX = 0.25          # 학습 미니배치 중 엔진 판의 비율 (엔진에게 배운 것을 잊지 않게)
REPLAY = 8
ITERATIONS = 200
# 3. 실력 측정
EVAL_EVERY = 5             # 전술 상대
EVAL_GAMES = 256
ENGINE_EVAL_EVERY = 10     # Rust 엔진 상대
ENGINE_EVAL_GAMES = 64
ENGINE_EVAL_TIME_MS = 20
SEED = 0
SAVE_DIR = "/content/drive/MyDrive/slider_results/alphazero"

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)
REPO = "/content/Slider"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, "https://github.com/Perdume/Slider.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mctx", "optax"], check=True)
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal", shell=True, check=True)
subprocess.run(["cargo", "build", "--release", "-q", "--bin", "selfplay", "--bin", "bestmove"], cwd=f"{REPO}/rust", check=True)
BIN = f"{REPO}/rust/target/release"
sys.path.insert(0, f"{REPO}/gpu")
import numpy as np
import jax, jax.numpy as jnp
import alphazero as az, engine_bridge as eb
importlib.reload(az); importlib.reload(eb)
print("device:", jax.devices(), "| CPU 코어:", os.cpu_count())

## 1. 엔진의 판 (없으면 만들고 Drive 에 저장)

In [ ]:
eng_files = []
for n_games, seed in ENGINE_SETS:
    name = f"engine_{n_games}g_{ENGINE_TIME_MS}ms.jsonl" if seed == 1 else f"engine_{n_games}g_{ENGINE_TIME_MS}ms_s{seed}.jsonl"
    path = os.path.join(SAVE_DIR, name)
    if not os.path.exists(path):
        t0 = time.time()
        tmp = f"/content/{name}"
        subprocess.run([f"{BIN}/selfplay", "--games", str(n_games), "--time-ms", str(ENGINE_TIME_MS), "--opts", "lmr2",
                        "--weights", f"{REPO}/weights/tuned.json", "--max-plies", "1000", "--random-plies", "4", "--epsilon", "0.05",
                        "--threads", str(os.cpu_count()), "--seed", str(seed), "--positions", "yes", "--out", tmp], check=True)
        shutil.copy(tmp, path)
        print(f"엔진 판 {n_games}개 만듦 ({time.time() - t0:.0f}초): {name}")
    eng_files.append(path)
E_obs, E_act, E_z = eb.load_engine_data(eng_files)
print("엔진 판 국면:", len(E_act))


## 2. 신경망 준비 (저장 파일이 있으면 이어서, 없으면 엔진의 판으로 사전학습)

In [ ]:
TAG = f"conv_c{CHANNELS}b{BLOCKS}"
CKPT = os.path.join(SAVE_DIR, f"ckpt_{TAG}.pkl")
opt = az.make_optimizer(LR)
engine = eb.Engine(f"{BIN}/bestmove", time_ms=ENGINE_EVAL_TIME_MS, threads=os.cpu_count())
if os.path.exists(CKPT):
    ck = az.load(CKPT)
    params = jax.tree_util.tree_map(jnp.asarray, ck["params"])
    opt_state = jax.tree_util.tree_map(jnp.asarray, ck["opt_state"])
    start_it, history = ck["iteration"], ck["history"]
    print(f"이어서 학습: 반복 {start_it} 부터")
else:
    params = az.make_params(jax.random.PRNGKey(SEED), arch=ARCH, channels=CHANNELS, blocks=BLOCKS)
    print("신경망 크기:", sum(x.size for x in jax.tree_util.tree_leaves(params)))
    opt_state = opt.init(params)
    print("엔진의 판으로 사전학습")
    params, opt_state = eb.pretrain(params, opt_state, opt, E_obs, E_act, E_z, PRETRAIN_STEPS, MINIBATCH, SEED)
    acc = eb.policy_accuracy(params, E_obs, E_act)
    w, d, l = eb.evaluate_vs_engine(params, engine, ENGINE_EVAL_GAMES, SIMULATIONS, MAX_LEN, DRAW_THRESHOLD, SEED)
    print(f"사전학습 끝: 엔진 수 맞히기 {acc:.1%} | 엔진 상대 {w}승 {d}무 {l}패 ({(w + 0.5 * d) / ENGINE_EVAL_GAMES:.2f})")
    start_it, history = 0, [{"iteration": 0, "pretrain_acc": acc, "vs_engine": [w, d, l],
                             "vs_engine_score": (w + 0.5 * d) / ENGINE_EVAL_GAMES}]
    az.save(CKPT, params, opt_state, 0, history)
replay = []

## 3. 강화학습 (멈춰도 됨)

In [ ]:
rng = np.random.default_rng(SEED + start_it)
key = jax.random.PRNGKey(SEED + 1000 + start_it)
for it in range(start_it, ITERATIONS):
    t0 = time.time()
    key, k_play, k_eval = jax.random.split(key, 3)
    obs, pi, active, winner = az.self_play(params, k_play, batch=GAMES, num_simulations=SIMULATIONS, max_len=MAX_LEN,
                                           draw_threshold=DRAW_THRESHOLD)
    obs, pi, active, winner = (np.asarray(x) for x in (obs, pi, active, winner))
    z = az.value_targets(active, winner)
    replay.append((obs[active].astype(np.uint8), pi[active].astype(np.float16), z[active]))
    replay = replay[-REPLAY:]
    X = np.concatenate([r[0] for r in replay]); P = np.concatenate([r[1] for r in replay]); Z = np.concatenate([r[2] for r in replay])
    n_eng = int(MINIBATCH * ENGINE_MIX)
    losses = []
    for _ in range(TRAIN_STEPS):
        i = rng.integers(0, len(Z), size=MINIBATCH - n_eng)
        j = rng.integers(0, len(E_z), size=n_eng)
        xb = jnp.asarray(np.concatenate([X[i], E_obs[j]]), jnp.float32)
        pb = jnp.concatenate([jnp.asarray(P[i], jnp.float32), jax.nn.one_hot(jnp.asarray(E_act[j]), az.A)])
        zb = jnp.asarray(np.concatenate([Z[i], E_z[j]]))
        params, opt_state, loss, (pl, vl) = az.train_step(params, opt_state, xb, pb, zb, opt)
        losses.append((float(pl), float(vl)))
    pl, vl = np.mean(losses, axis=0)
    row = {"iteration": it + 1, "avg_len": float(active.sum(0).mean()), "first_win": float((winner == 0).mean()),
           "second_win": float((winner == 1).mean()), "draw": float((winner < 0).mean()), "policy_loss": float(pl),
           "value_loss": float(vl)}
    if (it + 1) % EVAL_EVERY == 0:
        w, d, l = (int(x) for x in az.evaluate(params, k_eval, batch=EVAL_GAMES, num_simulations=SIMULATIONS, max_len=MAX_LEN,
                                               opponent="greedy", draw_threshold=DRAW_THRESHOLD))
        row["vs_greedy"], row["vs_greedy_score"] = [w, d, l], (w + 0.5 * d) / EVAL_GAMES
    if (it + 1) % ENGINE_EVAL_EVERY == 0:
        w, d, l = eb.evaluate_vs_engine(params, engine, ENGINE_EVAL_GAMES, SIMULATIONS, MAX_LEN, DRAW_THRESHOLD, SEED + it)
        row["vs_engine"], row["vs_engine_score"] = [w, d, l], (w + 0.5 * d) / ENGINE_EVAL_GAMES
    row["sec"] = time.time() - t0
    history.append(row)
    msg = (f"[{it + 1}] {row['sec']:.0f}s | 평균 {row['avg_len']:.0f}수, 선승 {row['first_win']:.0%} 후승 {row['second_win']:.0%} "
           f"무 {row['draw']:.0%} | 손실 정책 {pl:.3f} 가치 {vl:.3f}")
    if "vs_greedy" in row:
        msg += f" | 전술 상대 {row['vs_greedy']} ({row['vs_greedy_score']:.2f})"
    if "vs_engine" in row:
        msg += f" | Rust 엔진 {row['vs_engine']} ({row['vs_engine_score']:.2f})"
    print(msg, flush=True)
    az.save(CKPT, params, opt_state, it + 1, history)
    with open(os.path.join(SAVE_DIR, f"history_{TAG}.json"), "w") as f:
        json.dump(history, f, ensure_ascii=False, indent=1)

## 그래프

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h = [r for r in history if "policy_loss" in r]
ax[0].plot([r["iteration"] for r in h], [r["policy_loss"] for r in h], label="policy"); ax[0].plot([r["iteration"] for r in h], [r["value_loss"] for r in h], label="value"); ax[0].legend(); ax[0].set_title("loss")
for k, name in (("vs_greedy_score", "vs greedy"), ("vs_engine_score", "vs Rust engine")):
    e = [r for r in history if k in r]
    ax[1].plot([r["iteration"] for r in e], [r[k] for r in e], marker="o", label=name)
ax[1].axhline(0.5, color="gray", lw=0.5); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].set_title("eval score")
plt.show()